## Settings & imports

In [1]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from matplotlib.pyplot import cm
import string
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize
import csv
import pickle
import re
from scipy.spatial.distance import directed_hausdorff

In [2]:
dataset = 'XRF'
book_name = 'ml'

In [3]:
import json

with open('../config.json', 'r') as f:
    config = json.load(f)

which_dataset = config["which_dataset"]
preprocessing_method = config["preprocessing_method"] # normalization / logarithm
remove_outer_bool = config["remove_outer"]
how_many_outer_to_remove = config["how_many_outer_to_remove"]
elements_to_keep = config["elements_to_keep"][which_dataset]
elements_to_keep_xrf = config["elements_to_keep_xrf"]

data_path = config["data_path"][which_dataset]
target_path = config["target_path"][which_dataset]
figures_path = config["figures_path"]
models_path = config["models_path"][which_dataset]
classes_path = config["classes_path"]
Konstytucja_results_path = config["Konstytucja_results_path"][which_dataset]
xrf_path = config["xrf_path"]

results_path = config["results_path"]

## Loading the data & preprocessing

In [4]:
if dataset == 'Konstytucja_indicators':
    input_data = np.loadtxt(target_path, delimiter=',', skiprows=1, usecols=range(19))
    colnames = pd.read_csv(target_path, nrows=1, header=None)
    df = pd.DataFrame(data=input_data, columns=colnames.iloc[0,:-1])
    df = df[elements_to_keep]
elif dataset == 'Konstytucja_prediction':
    input_data = np.loadtxt(Konstytucja_results_path, delimiter=',')
    df = pd.DataFrame(data=input_data, columns=elements_to_keep)
elif dataset == 'XRF':
    input_data = np.loadtxt(xrf_path, delimiter=',', skiprows=1, usecols=(2,3,4))
    df = pd.DataFrame(data=input_data, columns=elements_to_keep_xrf)

df.reset_index(drop=True, inplace=True)

In [5]:
classes_df = pd.read_excel(classes_path, header=1, usecols=['NAZWA', 'OPIS'])
classes_df['NAZWA_short'] = classes_df['NAZWA'].apply(lambda x: 
                                                    re.split(r'(\d+)', x)[0] + re.split(r'(\d+)', x)[1] if len(re.split(r'(\d+)', x))>1 else re.split(r'(\d+)', x)[0]
                                                   )                                                    
classes_df.drop_duplicates(['NAZWA', 'OPIS'], inplace=True)
classes_df_supp = classes_df.drop_duplicates(['NAZWA_short'])
classes_df_supp['NAZWA'] = classes_df['NAZWA_short']
classes_df = pd.concat([classes_df, classes_df_supp])
classes_df.drop_duplicates(inplace=True)
classes_df.reset_index(inplace=True, drop=True)

/tmp/ipykernel_65221/3014040627.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  classes_df_supp['NAZWA'] = classes_df['NAZWA_short']


In [6]:
def remove_outer(group, n):
    return group.iloc[n:-n] if len(group) > 2*n else pd.DataFrame(columns=group.columns)

In [7]:
########################################################

In [8]:
if dataset == 'Konstytucja_indicators' or dataset == 'Konstytucja_prediction':
    
    if which_dataset == 'old':
        ground_truth_df = pd.read_csv(target_path, usecols=['probka', 'short'])
        ground_truth_df.rename(columns={"probka": "Sample_id"}, inplace=True)
    elif which_dataset == 'new':
        ground_truth_df = pd.read_csv(target_path, usecols=['name'])
        ground_truth_df['Sample_id'] = ground_truth_df['name'].apply(lambda x: x.split('_')[0] if len(x.split('_'))==2 else x.split('_')[0] + x.split('_')[1])
        ground_truth_df['Sample_id'] = ground_truth_df['Sample_id'].apply(lambda x: x.replace('.', ''))
        ground_truth_df['short'] = ground_truth_df['Sample_id'].apply(lambda x: re.split(r'\d+', x)[0])
        ground_truth_df.drop(columns=['name'], inplace=True)
    
    ground_truth_df.reset_index(drop=True, inplace=True)
    
    if remove_outer_bool:
        ground_truth_df = ground_truth_df.groupby('Sample_id', group_keys=False).apply(remove_outer, n=how_many_outer_to_remove)

elif dataset == 'XRF':

    ground_truth_df = pd.read_csv(xrf_path, usecols=['name'])
    ground_truth_df.rename(columns={'name': 'Sample_id'}, inplace=True)
    ground_truth_df['short'] = ground_truth_df['Sample_id'].apply(lambda x: re.split(r'\d+', x)[0])
    

In [9]:
ground_truth_df = pd.merge(left=ground_truth_df, right=classes_df, how='left', left_on='Sample_id', right_on='NAZWA')
    
ground_truth_df['OPIS'] = ground_truth_df['OPIS'].apply(lambda x: str(x).strip())
ground_truth_df.drop(columns=['NAZWA', 'NAZWA_short'], inplace=True)
ground_truth_df.reset_index(drop=True, inplace=True)

### Dividing to APP, ASC, ML

In [10]:
df_app = df[ground_truth_df['short'].apply(lambda x: x.startswith('APP'))]
df_asc = df[ground_truth_df['short'].apply(lambda x: x.startswith('ASC'))]
df_ml = df[ground_truth_df['short'].apply(lambda x: x.startswith('ML'))]

if book_name == 'app':
    df = df_app
elif book_name == 'asc':
    df = df_asc
elif book_name == 'ml':
    df = df_ml
elif book_name == 'all':
    pass

### Removing some data

1. Let's keep only columns that we need.

To reduce the set of used elements run cell below. Then, instead of predicting 29 numbers, we will predict only 8. We will also use only 8 numbers as input.

In [11]:
if dataset == 'Konstytucja_indicators' or dataset == 'Konstytucja_prediction':
    df = df[elements_to_keep]

2. Let's check if there are any rows with missing values.

In [12]:
(df.shape[0] - df.dropna().shape[0])/df.shape[0]

0.0

3. Let's reset index.

In [13]:
df.reset_index(drop=True, inplace=True)

### Converting to np.array

In [14]:
X = np.array(df.values)

### Normalizing / taking logarithm

In [15]:
def adjusted_log_transform(input_array):
    res = np.where(input_array>0, np.log(input_array), 0.)
    return res

In [16]:
if dataset == 'Konstytucja_indicators' or dataset == 'Konstytucja_prediction':

    if preprocessing_method == 'normalization':
    
        X = (X - np.min(X, axis=0))/np.std(X, axis=0)
        
    elif preprocessing_method == 'logarithm':
        
        X = adjusted_log_transform(X)
    
    elif preprocessing_method == 'logarithm_and_normalization':
        
        #logarithm
        X = adjusted_log_transform(X)
        
        #normalization
        X = (X - np.min(X, axis=0))/np.std(X, axis=0)
        
    elif preprocessing_method == 'none':
        
        pass

### Ground truth

In [17]:
class_df_app = ground_truth_df[ground_truth_df['short'].apply(lambda x: x.startswith('APP'))][['Sample_id','OPIS']]
class_df_asc = ground_truth_df[ground_truth_df['short'].apply(lambda x: x.startswith('ASC'))][['Sample_id', 'OPIS']]
class_df_ml = ground_truth_df[ground_truth_df['short'].apply(lambda x: x.startswith('ML'))][['Sample_id', 'OPIS']]

if book_name == 'app':
    class_df = class_df_app
elif book_name == 'asc':
    class_df = class_df_asc
elif book_name == 'ml':
    class_df = class_df_ml
elif book_name == 'all':
    class_df = ground_truth_df[['Sample_id', 'OPIS']]

class_df.columns = ['Code', 'Class name']

In [18]:
y = class_df['Class name']
y = [str(y) for y in y]
class_df.reset_index(inplace=True, drop=True)

### Closest classes (point by point, only pairs with signatures)

In [19]:
non_signatures_df = df[class_df['Class name'].apply(lambda x: 'podpis' not in x)]
non_signatures_class_df = class_df[class_df['Class name'].apply(lambda x: 'podpis' not in x)]

In [20]:
n_smallest = 2
closest = []
for i in range(non_signatures_df.shape[0]):
    which_row = ((abs(non_signatures_df.iloc[i, :] - df)).sum(1)).nsmallest(n_smallest).index[1]
    closest.append((which_row, class_df.iloc[which_row]['Code'], class_df.iloc[which_row]['Class name']))

In [21]:
non_signatures_class_df[['Closest index', 'Closest code', 'Closest class name']] = closest

/tmp/ipykernel_65221/937534572.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  non_signatures_class_df[['Closest index', 'Closest code', 'Closest class name']] = closest
/tmp/ipykernel_65221/937534572.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  non_signatures_class_df[['Closest index', 'Closest code', 'Closest class name']] = closest
/tmp/ipykernel_65221/937534572.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_inde

In [22]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None): 
    print(non_signatures_class_df.sort_values('Code'))

              Code Class name Closest index    Closest code  \
15           ML14A   poprawka            23           ML16C   
16           ML14B   poprawka            56         MLMAZC3   
17           ML14C   poprawka            15           ML14A   
18           ML15A      treść            20           ML15C   
19           ML15B      treść             7           ML11B   
20           ML15C      treść            21           ML16A   
21           ML16A      treść            10           ML12B   
22           ML16B      treść             3           ML10A   
23           ML16C      treść            15           ML14A   
51  MLMALACHOWSKI1        nan            61         MLMAZS2   
52  MLMALACHOWSKI2        nan            68      MLSAPIEHA3   
53  MLMALACHOWSKI3        nan            59         MLMAZG3   
54         MLMAZC1        nan            55         MLMAZC2   
55         MLMAZC2        nan            54         MLMAZC1   
56         MLMAZC3        nan            62         MLM

APP14 Jabłonowski - według Konstytucja_prediction (2/3)
APP15A Puttkamer - według Konstytucja_prediction (3/5)

APP15D Sapieha - według Konstytucja_indicators (5/8)

ML15C Nowowiejski - według Konstytucja_prediction (4/5)
ML16B Nowowiejski - według Konstytucja_prediction (2/3)

ML15B Nowowiejski - według Konstytucja_indicators (2/2)
ML15C Nowowiejski - według Konstytucja_indicators (4/5)
ML16C Potocki - według Konstytucja_indicators (2/2)

ASC14 Małachowski - według Konstytucja_prediction (2/2)
ASC2A Małachowski - według Konstytucja_prediction (2/2)
ASC3B Sapieha - według Konstytucja_prediction (2/2)
ASC5A Sapieha - według Konstytucja_prediction (3/3)

ASC0B Małachowski - według Konstytucja_indicators (3/3)
ASC11C Sapieha - według Konstytucja_indicators (2/2)
ASC13A Małachowski - według Konstytucja_indicators (2/2)
ASC15C Małachowski - według Konstytucja_indicators (2/2)
ASC16C Małachowski - według Konstytucja_indicators (3/3)
ASC18A Sapieha - według Konstytucja_indicators (2/2)
ASC3A, ASC3C Sapieha - według Konstytucja_indicators (1/1 każdy)
ASC4A Małachowski - według Konstytucja_indicators (3/3)
ASC5A Sapieha - według Konstytucja_indicators (7/7)

ASC4 Małachowski - według XRF (2/2)

### Closest classes (set to set, Hausdorff distance, only pairs with signatures)

In [23]:
n_smallest = 2

closest_sets = pd.DataFrame(columns = ['No signatures class', 'Closest class', 'Hausdorff distance'])

for chosen_class in non_signatures_class_df['Code'].unique():
    
    chosen_class_distances = pd.DataFrame(columns=['Class', 'Hausdorff distance'])
    
    for code in class_df['Code'].unique():
        
        set1 = non_signatures_df[non_signatures_class_df['Code'] == chosen_class].values
        set2 = df[class_df['Code'] == code].values
        chosen_class_distances = pd.concat([chosen_class_distances, 
                                            pd.DataFrame([[code, directed_hausdorff(set1, set2)[0]]],
                                                        columns=['Class', 'Hausdorff distance'])])
        chosen_class_distances.reset_index(inplace=True, drop=True)
        
    which_row = chosen_class_distances.nsmallest(n_smallest, 'Hausdorff distance').index[1]
    closest_sets = pd.concat([closest_sets, 
                             pd.DataFrame([[chosen_class, 
                                            chosen_class_distances.iloc[which_row]['Class'], 
                                            chosen_class_distances.iloc[which_row]['Hausdorff distance']]],
                                             columns = ['No signatures class', 'Closest class', 'Hausdorff distance']
                                         )])

/tmp/ipykernel_65221/2973331564.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  chosen_class_distances = pd.concat([chosen_class_distances,
/tmp/ipykernel_65221/2973331564.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  closest_sets = pd.concat([closest_sets,
/tmp/ipykernel_65221/2973331564.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining th

In [24]:
closest_sets = closest_sets.merge(class_df.drop_duplicates(), 
                                  how='left', 
                                  left_on = 'No signatures class', 
                                  right_on='Code')
closest_sets.rename(columns = {'Class name': 'No signatures class name'},
                    inplace=True)
closest_sets.drop(columns='Code', inplace=True)
closest_sets.reset_index(inplace=True, drop=True)


closest_sets = closest_sets.merge(class_df.drop_duplicates(), 
                                  how='left', 
                                  left_on = 'Closest class', 
                                  right_on='Code')
closest_sets.rename(columns = {'Class name': 'Closest class name'},
                    inplace=True)
closest_sets.drop(columns='Code', inplace=True)

closest_sets.sort_values(by='Hausdorff distance', inplace=True)
closest_sets.reset_index(inplace=True, drop=True)

In [25]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None): 
    print(closest_sets)

   No signatures class   Closest class  Hausdorff distance  \
0                ML16A           ML12B            0.006441   
1       MLMALACHOWSKI3         MLMAZG3            0.009156   
2              MLMAZG3  MLMALACHOWSKI3            0.009156   
3              MLMAZG1         MLMAZG2            0.009751   
4              MLMAZG2         MLMAZG1            0.009751   
5               MLMF1B          MLMF1C            0.012701   
6               MLMF1C          MLMF1B            0.012701   
7       MLMALACHOWSKI1         MLMAZS2            0.013625   
8              MLMAZS2  MLMALACHOWSKI1            0.013625   
9               MLMF1A            ML0C            0.014789   
10             MLMAZC3         MLMAZS3            0.017328   
11             MLMAZS3         MLMAZC3            0.017328   
12               ML16C           ML14A            0.017820   
13               ML14A           ML16C            0.017820   
14          MLSAPIEHA2      MLSAPIEHA3            0.019889   
15      

In [26]:
closest_sets[closest_sets['Closest class name'].apply(lambda x: 'podpis' in x)]
# Konstytucja indicators APP

,No signatures class,Closest class,Hausdorff distance,No signatures class name,Closest class name
0,APP15B,APP9B,169.423480,poprawka,podpis - Nowowiejski
2,APP16B,APP7A,217.768757,oblatowanie,podpis - Potocki
3,APP15C,APP9A,227.417831,poprawka,podpis - Nowowiejski
7,APP16A,APP12B,474.065054,oblatowanie,podpis - Puttkamer
8,APP14,APP2,556.085505,treść,podpis - Jabłonowski
13,APP19,APP10,1275.479269,treść,podpis - Radzicki
15,APP15D,APP0C,2154.277559,poprawka,podpis - Małachowski


In [26]:
closest_sets[closest_sets['Closest class name'].apply(lambda x: 'podpis' in x)]
# Konstytucja prediction APP

,No signatures class,Closest class,Hausdorff distance,No signatures class name,Closest class name
0,APP14,APP2,0.429483,treść,podpis - Jabłonowski
1,APP18,APP1,0.496449,poprawka,podpis - Kossakowski
2,APP20,APP8,0.524613,treść,podpis - Zboiński
3,APP19,APP11,0.574653,treść,podpis - Zabiełło
4,APP15,APP2,0.599153,poprawka,podpis - Jabłonowski
6,APP15C,APP9B,0.741489,poprawka,podpis - Nowowiejski
9,APP15A,APP5B,0.979163,poprawka,podpis - Plater
12,APP16C,APP6A,1.227591,oblatowanie,podpis - Stroynowski
13,APP16B,APP6B,1.446561,oblatowanie,podpis - Stroynowski
15,APP22,APP6B,1.582969,treść,podpis - Stroynowski


In [26]:
closest_sets[closest_sets['Closest class name'].apply(lambda x: 'podpis' in x)]
# XRF APP

,No signatures class,Closest class,Hausdorff distance,No signatures class name,Closest class name
4,APPMF1B,APP9A,0.011042,nan,podpis - Nowowiejski
5,APP15B1,APP6C,0.016142,poprawka,podpis - Stroynowski
6,APP14D,APP7A,0.016538,treść,podpis - Potocki
8,APP14A,APP12C,0.024149,treść,podpis - Puttkamer
9,APPMF1C,APP3A,0.024624,nan,podpis - Szydłowski
10,APPMF1A,APP3A,0.032850,nan,podpis - Szydłowski
11,APP15A1,APP5A,0.051084,poprawka,podpis - Plater


In [26]:
closest_sets[closest_sets['Closest class name'].apply(lambda x: 'podpis' in x)]
# Konstytucja indicators ML

,No signatures class,Closest class,Hausdorff distance,No signatures class name,Closest class name
2,ML16A,ML5C,225.515230,treść,podpis - Plater
4,ML16B,ML8C,303.809069,treść,podpis - Zboiński
5,ML15B,ML10C,403.137427,treść,podpis - Radzicki
6,ML15C,ML9B,482.485973,treść,podpis - Nowowiejski
7,ML14A,ML1C,2353.100858,poprawka,podpis - Kossakowski
8,ML14B,ML4B,2677.555711,poprawka,podpis - Kwilecki


In [26]:
closest_sets[closest_sets['Closest class name'].apply(lambda x: 'podpis' in x)]
# Konstytucja prediction ML

,No signatures class,Closest class,Hausdorff distance,No signatures class name,Closest class name
0,ML16A,ML5C,0.371786,treść,podpis - Plater
1,ML16C,ML7C,0.463946,treść,podpis - Potocki
2,ML14C,ML5C,0.555196,poprawka,podpis - Plater
4,ML14A,ML9C,0.634642,poprawka,podpis - Nowowiejski
6,ML15A,ML8A,0.839463,treść,podpis - Zboiński
7,ML15C,ML11A,0.915399,treść,podpis - Zabiełło
8,ML15B,ML0B,0.986246,treść,podpis - Małachowski


In [26]:
closest_sets[closest_sets['Closest class name'].apply(lambda x: 'podpis' in x)]
# XRF ML

,No signatures class,Closest class,Hausdorff distance,No signatures class name,Closest class name
0,ML16A,ML12B,0.006441,treść,podpis - Puttkamer
9,MLMF1A,ML0C,0.014789,nan,podpis - Małachowski
19,ML15B,ML11B,0.023024,treść,podpis - Zabiełło
23,ML16B,ML10A,0.034649,treść,podpis - Radzicki
24,ML15C,ML12B,0.044829,treść,podpis - Puttkamer
26,MLSAPIEHA1,ML2C,0.125846,nan,podpis - Jabłonowski


In [26]:
closest_sets[closest_sets['Closest class name'].apply(lambda x: 'podpis' in x)]
# Konstytucja indicators ASC

,No signatures class,Closest class,Hausdorff distance,No signatures class name,Closest class name
5,ASC10A,ASC20A,74.821765,treść,podpis - Małachowski
26,ASC3A,ASC21A,122.885391,komentarz,podpis - Sapieha
30,ASC13C,ASC21B,131.040213,komentarz,podpis - Sapieha
81,ASC2,ASC25,1257.089036,poprawka,podpis - Małachowski
85,ASC14,ASC27,4559.551892,komentarz,podpis - Małachowski
86,ASC0B,ASC20,5162.383575,oblatowanie,podpis - Małachowski


In [26]:
closest_sets[closest_sets['Closest class name'].apply(lambda x: 'podpis' in x)]
# Konstytucja prediction ASC

,No signatures class,Closest class,Hausdorff distance,No signatures class name,Closest class name
3,ASC14,ASC27,0.476316,komentarz,podpis - Małachowski
23,ASC3B,ASC21B,0.987836,komentarz,podpis - Sapieha
27,ASC13,ASC26,1.013456,komentarz,podpis - Małachowski
31,ASC9B,ASC21B,1.059200,komentarz,podpis - Sapieha
38,ASC3A,ASC21A,1.140662,komentarz,podpis - Sapieha
44,ASC2,ASC20,1.198436,poprawka,podpis - Małachowski
49,ASC11A,ASC21B,1.264436,komentarz,podpis - Sapieha
54,ASC11C,ASC21B,1.407958,komentarz,podpis - Sapieha
56,ASC19A2,ASC20C,1.427047,treść,podpis - Małachowski
65,ASC14B,ASC27,1.539591,komentarz,podpis - Małachowski


In [26]:
closest_sets[closest_sets['Closest class name'].apply(lambda x: 'podpis' in x)]
# XRF ASC

,No signatures class,Closest class,Hausdorff distance,No signatures class name,Closest class name
27,ASC2C,ASC20C,0.014955,poprawka,podpis - Małachowski
36,ASC4C,ASC20B,0.021510,treść,podpis - Małachowski
56,ASC4,ASC20B,0.038960,treść,podpis - Małachowski
66,ASC22,ASC21,1.611424,plama,podpis - Sapieha


### Closest classes (set to set, Hausdorff distance, all pairs)

In [27]:
n_smallest = 2

closest_sets = pd.DataFrame(columns = ['Class', 'Closest class', 'Hausdorff distance'])

for chosen_class in class_df['Code'].unique():
    
    chosen_class_distances = pd.DataFrame(columns=['Class', 'Hausdorff distance'])
    
    for code in class_df['Code'].unique():
        
        set1 = df[class_df['Code'] == chosen_class].values
        set2 = df[class_df['Code'] == code].values
        chosen_class_distances = pd.concat([chosen_class_distances, 
                                            pd.DataFrame([[code, directed_hausdorff(set1, set2)[0]]],
                                                        columns=['Class', 'Hausdorff distance'])])
        chosen_class_distances.reset_index(inplace=True, drop=True)
        
    which_row = chosen_class_distances.nsmallest(n_smallest, 'Hausdorff distance').index[1]
    closest_sets = pd.concat([closest_sets, 
                             pd.DataFrame([[chosen_class, 
                                            chosen_class_distances.iloc[which_row]['Class'], 
                                            chosen_class_distances.iloc[which_row]['Hausdorff distance']]],
                                             columns = ['Class', 'Closest class', 'Hausdorff distance']
                                         )])

/tmp/ipykernel_65221/667569848.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  chosen_class_distances = pd.concat([chosen_class_distances,
/tmp/ipykernel_65221/667569848.py:19: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  closest_sets = pd.concat([closest_sets,
/tmp/ipykernel_65221/667569848.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the r

In [28]:
closest_sets = closest_sets.merge(class_df.drop_duplicates(), 
                                  how='left', 
                                  left_on = 'Class', 
                                  right_on='Code')
# closest_sets.rename(columns = {'Class name': 'No signatures class name'},
#                     inplace=True)
closest_sets.drop(columns='Code', inplace=True)
closest_sets.reset_index(inplace=True, drop=True)


closest_sets = closest_sets.merge(class_df.drop_duplicates(), 
                                  how='left', 
                                  left_on = 'Closest class', 
                                  right_on='Code')
closest_sets.rename(columns = {'Class name_x': 'Class name', 'Class name_y': 'Closest class name'},
                    inplace=True)
closest_sets.drop(columns='Code', inplace=True)

closest_sets.sort_values(by='Hausdorff distance', inplace=True)
closest_sets.reset_index(inplace=True, drop=True)
closest_sets.sort_values(by=['Class'], inplace=True)

In [29]:
with pd.ExcelWriter(results_path + 'Hausdorff_closest_classes.xlsx', mode='a') as writer:  
    closest_sets.to_excel(writer, sheet_name = str.upper(book_name) + '_' + dataset, index=False)